# WANDS: full fine-tune of all-MiniLM-L6-v2 (item 8)

The one training run from the build plan that runs on Colab instead of the 8 GB MacBook: the full fine-tune (every weight trained, no LoRA). Training pairs, targets, seed, holdout and hyperparameters are identical to `src/train_lora.py` in the repo, so this run is comparable to the local LoRA runs. Only the hardware differs (Colab GPU vs M2 CPU); training time is recorded but not compared directly against the CPU runs.

Needs a GPU runtime (Runtime > Change runtime type > T4 GPU). Produces `full_ft_colab.zip` (the merged sentence-transformers model plus `metrics.json`) and downloads it. Put the unzipped model at `data/models/full_ft_merged/` in the repo.

In [ ]:
%pip -q install sentence-transformers==3.4.1 datasets accelerate
!git clone -q --depth 1 https://github.com/wayfair/WANDS.git wands_data
print("setup done")

In [ ]:
# Pair building: identical to src/train_lora.py (seed 0, same strata and counts).
import hashlib, random, time
import pandas as pd

PRODUCTS_CSV = "wands_data/dataset/product.csv"
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SEED = 0
N_SAME_PATH = 10000
N_SAME_ROOT = 10000
N_CROSS = 10000
N_EVAL = 3000

def product_rows():
    p = pd.read_csv(PRODUCTS_CSV, sep="\t").fillna("")
    p = p.rename(columns={"category hierarchy": "category_hierarchy"})
    text = (p.product_name + ". " + p.product_class.str.replace("|", ", ", regex=False)
            + ". " + p.category_hierarchy.str.replace("/", ", ", regex=False)
            + ". " + p.product_description)
    path = [tuple(s.strip() for s in h.split("/") if s.strip())
            for h in p.category_hierarchy]
    return list(p.product_id), text.tolist(), path

def similarity(a, b):
    if not a or not b:
        return 0.0
    shared = 0
    for x, y in zip(a, b):
        if x != y:
            break
        shared += 1
    return shared / max(len(a), len(b))

def holdout(pid):
    return int(hashlib.md5(str(pid).encode()).hexdigest(), 16) % 10 == 0

def sample_pairs(idx, texts, paths, n_same_path, n_same_root, n_cross, rng):
    by_path, by_root = {}, {}
    for i in idx:
        if paths[i]:
            by_path.setdefault(paths[i], []).append(i)
            by_root.setdefault(paths[i][0], []).append(i)
    multi_path = [g for g in by_path.values() if len(g) > 1]
    multi_root = [g for g in by_root.values() if len(g) > 1]
    pairs = []
    for _ in range(n_same_path):
        g = rng.choice(multi_path)
        pairs.append(tuple(rng.sample(g, 2)))
    for _ in range(n_same_root):
        g = rng.choice(multi_root)
        pairs.append(tuple(rng.sample(g, 2)))
    for _ in range(n_cross):
        pairs.append(tuple(rng.sample(idx, 2)))
    return {"sentence1": [texts[a] for a, _ in pairs],
            "sentence2": [texts[b] for _, b in pairs],
            "score": [similarity(paths[a], paths[b]) for a, b in pairs]}

rng = random.Random(SEED)
ids, texts, paths = product_rows()
train_idx = [i for i, pid in enumerate(ids) if not holdout(pid)]
eval_idx = [i for i, pid in enumerate(ids) if holdout(pid)]
train_data = sample_pairs(train_idx, texts, paths, N_SAME_PATH, N_SAME_ROOT, N_CROSS, rng)
k = N_EVAL // 3
eval_data = sample_pairs(eval_idx, texts, paths, k, k, k, rng)
print(len(train_data["score"]), "train pairs,", len(eval_data["score"]), "eval pairs")

In [ ]:
# Full fine-tune: every weight trains. Same loss, batch size, epochs and seed as the LoRA runs.
import torch
from datasets import Dataset
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sentence_transformers.losses import CosineSimilarityLoss

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch the runtime to a GPU first (Runtime > Change runtime type)."
model = SentenceTransformer(BASE_MODEL, device=device)

args = SentenceTransformerTrainingArguments(
    output_dir="full_ft_trainer", num_train_epochs=1,
    per_device_train_batch_size=32, per_device_eval_batch_size=32,
    eval_strategy="no", save_strategy="no", logging_steps=100, seed=SEED, report_to=[],
)
trainer = SentenceTransformerTrainer(
    model=model, args=args,
    train_dataset=Dataset.from_dict(train_data),
    eval_dataset=Dataset.from_dict(eval_data),
    loss=CosineSimilarityLoss(model),
)
start = time.time()
result = trainer.train()
minutes = (time.time() - start) / 60
eval_loss = trainer.evaluate()["eval_loss"]
print(f"trained in {minutes:.1f} min, train loss {result.training_loss:.4f}, eval loss {eval_loss:.4f}")

In [ ]:
# Save, zip, download. Unzip into data/models/full_ft_merged/ in the repo.
import json, subprocess
from pathlib import Path

model.save("full_ft_merged")
metrics = {"mode": "full", "base_model": BASE_MODEL, "epochs": 1, "batch_size": 32,
           "seed": SEED, "train_pairs": len(train_data["score"]),
           "eval_pairs": len(eval_data["score"]), "loss": "CosineSimilarityLoss",
           "train_loss": result.training_loss, "eval_loss": eval_loss,
           "train_minutes": minutes, "device": torch.cuda.get_device_name(0),
           "saved_mb": sum(f.stat().st_size for f in Path("full_ft_merged").rglob("*") if f.is_file()) / 1e6}
Path("full_ft_merged/metrics.json").write_text(json.dumps(metrics, indent=2))
subprocess.run(["zip", "-qr", "full_ft_colab.zip", "full_ft_merged"], check=True)
print(json.dumps(metrics, indent=2))
from google.colab import files
files.download("full_ft_colab.zip")